In [ ]:
import os
import re
import pyspark
from dotenv import load_dotenv
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization
from pyspark.sql import functions as F
from sedona.spark import SedonaContext


def load_snowflake_private_key(key_path, passphrase=None):
    with open(key_path, "rb") as f:
        p_key = serialization.load_pem_private_key(
            f.read(),
            password=passphrase.encode() if passphrase else None,
            backend=default_backend(),
        )
    pkb = p_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    ).decode("utf-8")
    return re.sub(r"-*(BEGIN|END) PRIVATE KEY-*\n", "", pkb).replace("\n", "")


def build_spark_session(gcp_key_path=None):
    spark_mm = ".".join(pyspark.__version__.split(".")[:2])
    scala_ver = "2.13" if int(spark_mm.split(".")[0]) >= 4 else "2.12"

    packages = ",".join([
        f"org.apache.sedona:sedona-spark-{spark_mm}_{scala_ver}:1.9.1",
        "org.datasyslab:geotools-wrapper:1.9.1-33.5",
        f"net.snowflake:spark-snowflake_{scala_ver}:3.2.3-spark_{spark_mm}",
        "net.snowflake:snowflake-jdbc:4.4.0",
        "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.22",
    ])

    builder = (
        SedonaContext.builder()
        .appName("GeoPulse_Spatial_Linking_ETL")
        .config("spark.jars.packages", packages)
        .config(
            "spark.jars.repositories",
            "https://artifacts.unidata.ucar.edu/repository/unidata-all",
        )
        .config(
            "spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        )
        .config(
            "spark.hadoop.fs.AbstractFileSystem.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS",
        )
    )

    if gcp_key_path:
        builder = (
            builder.config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
            .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", gcp_key_path)
        )

    config = builder.getOrCreate()
    return SedonaContext.create(config)


def run_linking_etl(
    raw_gps_path,
    stores_list,
    sf_options,
    gcp_key_path=None,
    catchment_radius_m=500.0,
    visit_radius_m=20.0,
    min_pings=3,
    staff_dwell_threshold_min=240.0,
    write_mode="overwrite",
):
    sedona = build_spark_session(gcp_key_path=gcp_key_path)
    snowflake_source = "net.snowflake.spark.snowflake"

    clean_pings = None
    ping_store_matches = None

    try:
        # 1. Load raw GPS stream from GCS (Parquet by default, CSV fallback)
        if raw_gps_path.lower().endswith(".csv"):
            raw_df = sedona.read.csv(raw_gps_path, header=True, inferSchema=True)
        else:
            raw_df = sedona.read.parquet(raw_gps_path)

        clean_pings = (
            raw_df.select(
                F.col("DeviceID").cast("string").alias("device_id"),
                F.col("Latitude").cast("double").alias("lat"),
                F.col("Longitude").cast("double").alias("lon"),
                F.to_timestamp(F.col("Timestamp")).alias("ts"),
            )
            .filter(
                F.col("device_id").isNotNull()
                & F.col("ts").isNotNull()
                & F.col("lat").between(-90.0, 90.0)
                & F.col("lon").between(-180.0, 180.0)
            )
            .dropDuplicates(["device_id", "ts", "lat", "lon"])
            .withColumn("ping_pt", F.expr("ST_Point(lon, lat)"))
            .cache()
        )

        # 2. Build store catalog DataFrame
        stores_df = sedona.createDataFrame(stores_list).select(
            F.col("store_id").cast("string"),
            F.col("store_name").cast("string"),
            F.col("lat").cast("double").alias("store_lat"),
            F.col("lon").cast("double").alias("store_lon"),
            F.col("attractiveness").cast("double"),
        )
        stores_spatial = stores_df.withColumn(
            "store_pt", F.expr("ST_Point(store_lon, store_lat)")
        )

        # 3. Compute device activity centroids (for Huff Gravity in Snowflake)
        device_centroids = clean_pings.groupBy("device_id").agg(
            F.avg("lat").alias("avg_lat"),
            F.avg("lon").alias("avg_lon"),
            F.count("*").alias("total_pings"),
            F.countDistinct(F.to_date("ts")).alias("active_days"),
        )

        # 4. Spatial linking: fast bounding-box prefilter + exact spherical distance
        lat_pad = (catchment_radius_m / 111_000.0) * 1.25
        lon_pad = lat_pad * 1.35

        ping_store_matches = (
            clean_pings.join(
                F.broadcast(stores_spatial),
                on=(
                    (F.abs(F.col("lat") - F.col("store_lat")) <= F.lit(lat_pad))
                    & (F.abs(F.col("lon") - F.col("store_lon")) <= F.lit(lon_pad))
                ),
                how="inner",
            )
            .withColumn("dist_m", F.expr("ST_DistanceSphere(ping_pt, store_pt)"))
            .filter(F.col("dist_m") <= F.lit(catchment_radius_m))
            .select("store_id", "device_id", "ts", "dist_m")
            .cache()
        )

        # 5. Aggregate catchment presence per (store_id, device_id)
        catchment_agg = ping_store_matches.groupBy("store_id", "device_id").agg(
            F.count("*").alias("catchment_pings"),
            F.round(F.min("dist_m"), 2).alias("min_dist_m"),
            F.min("ts").alias("first_seen_ts"),
            F.max("ts").alias("last_seen_ts"),
        )

        # 6. Identify confirmed store visits & dwell time inside tight store radius
        visit_sessions = (
            ping_store_matches.filter(F.col("dist_m") <= F.lit(visit_radius_m))
            .groupBy("store_id", "device_id", F.window("ts", "1 hour"))
            .agg(
                F.count("*").alias("ping_count"),
                (
                    (F.unix_timestamp(F.max("ts")) - F.unix_timestamp(F.min("ts")))
                    / 60.0
                ).alias("session_dwell_min"),
            )
            .filter(F.col("ping_count") >= F.lit(min_pings))
        )

        visit_agg = visit_sessions.groupBy("store_id", "device_id").agg(
            F.count("*").alias("confirmed_visits"),
            F.round(F.sum("session_dwell_min"), 2).alias("total_dwell_min"),
            F.round(F.max("session_dwell_min"), 2).alias("max_session_dwell_min"),
        )

        # 7. Final linked table (plain primitives only, no Geometry UDTs)
        store_device_links = (
            catchment_agg.join(visit_agg, on=["store_id", "device_id"], how="left")
            .fillna({"confirmed_visits": 0, "total_dwell_min": 0.0, "max_session_dwell_min": 0.0})
            .withColumn("in_catchment", F.lit(True))
            .withColumn(
                "is_suspected_staff",
                F.col("max_session_dwell_min") >= F.lit(staff_dwell_threshold_min),
            )
            .select(
                "store_id",
                "device_id",
                "in_catchment",
                "catchment_pings",
                "confirmed_visits",
                "total_dwell_min",
                "min_dist_m",
                "is_suspected_staff",
                "first_seen_ts",
                "last_seen_ts",
            )
        )

        # 8. Write all three tables to Snowflake
        tables = {
            "STORES_DIM": stores_df,
            "DEVICE_CENTROIDS": device_centroids,
            "STORE_DEVICE_LINKS": store_device_links,
        }

        for table_name, df in tables.items():
            (
                df.write.format(snowflake_source)
                .options(**sf_options)
                .option("dbtable", table_name)
                .mode(write_mode)
                .save()
            )
            print(f"Wrote table {table_name} to Snowflake.")

    finally:
        if ping_store_matches is not None:
            ping_store_matches.unpersist()
        if clean_pings is not None:
            clean_pings.unpersist()
        sedona.stop()


if __name__ == "__main__":
    load_dotenv()

    private_key_pem = load_snowflake_private_key(
        key_path=os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"],
        passphrase=os.environ.get("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE"),
    )

    sf_options = {
        "sfURL": os.environ["SNOWFLAKE_URL"],
        "sfUser": os.environ["SNOWFLAKE_USER"],
        "pem_private_key": private_key_pem,
        "sfDatabase": "GEOPULSE_DB",
        "sfSchema": "ANALYTICS",
        "sfWarehouse": "COMPUTE_WH",
        "sfRole": "GEOPULSE_ETL_ROLE",
    }

    stores = [
        {"store_id": "BLR_001", "store_name": "Indiranagar 100ft Rd", "lat": 12.978973, "lon": 77.645270, "attractiveness": 1.0},
        {"store_id": "BLR_002", "store_name": "Indiranagar CMH Rd",   "lat": 12.973500, "lon": 77.641200, "attractiveness": 1.0},
        {"store_id": "BLR_003", "store_name": "Koramangala 4th Blk",  "lat": 12.935200, "lon": 77.624500, "attractiveness": 1.2},
    ]

    run_linking_etl(
        raw_gps_path=os.environ["GCS_RAW_GPS_PATH"],  # e.g. "gs://geopulse-raw-data/pings/*.parquet"
        stores_list=stores,
        sf_options=sf_options,
        gcp_key_path=os.environ["GOOGLE_APPLICATION_CREDENTIALS"],
        write_mode="overwrite",
    )